In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import patches
import seaborn as sns
from copy import deepcopy

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from modules.autolog import ASLT
from sklearn.preprocessing import StandardScaler,OrdinalEncoder,PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans,SpectralClustering,AgglomerativeClustering
from sklearn.model_selection import cross_val_score
from sklearn.cluster import DBSCAN
from sklearn.metrics import adjusted_rand_score,homogeneity_completeness_v_measure,silhouette_score
import optuna
from sklearn.base import clone

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [2]:
df = pd.read_csv('./abalone/abalone.data',header=None,names=['sex','length','diameter','height','whole_weight','shucked_weight','viscera_weight','shell_weight','rings'])
df['sex'] = df.pop('sex')
print(df.shape)
df.head()

(4177, 9)


,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,rings,sex
0,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15,M
1,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7,M
2,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9,F
3,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10,M
4,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7,I


In [3]:
print('NaN values of each columns:')
print(df.isna().sum())

NaN values of each columns:
length            0
diameter          0
height            0
whole_weight      0
shucked_weight    0
viscera_weight    0
shell_weight      0
rings             0
sex               0
dtype: int64


In [4]:
print('Count of duplicated rows:')
print(df.duplicated().sum())

Count of duplicated rows:
0


In [5]:
print('Datatypes of each columns:')
print(df.dtypes)

Datatypes of each columns:
length            float64
diameter          float64
height            float64
whole_weight      float64
shucked_weight    float64
viscera_weight    float64
shell_weight      float64
rings               int64
sex                object
dtype: object


# **Preprocessing**

### **Removing Outliers with DBSCAN**

In [6]:
pl_out = Pipeline([
    ('encoder',ColumnTransformer([
      ('cat',OrdinalEncoder(),['sex'])  
    ],remainder='passthrough')),
    ('scaler',StandardScaler()),
    ('cluster',DBSCAN(eps=1.5,min_samples=10))
])
outlier_indexs = pl_out.fit_predict(df)
outlier_indexs = np.where(outlier_indexs == -1, 1, 0).astype(bool)
df_out = df[~outlier_indexs]
print(f'Removed {outlier_indexs.sum()} rows detected as outliers.')

df_out = df_out.reset_index(drop=True)

Removed 24 rows detected as outliers.


# **Modelling**

### **Check for Sex**

In [7]:
df_sex = df_out.drop('sex',axis=1)
y_sex = df_out['sex']

num_cols = df_out.select_dtypes(include='number').columns
cat_cols = df_out.select_dtypes(exclude='number').columns
num_pl = Pipeline([
            ('aslt',ASLT(beta=10)),
            ('scaler',StandardScaler()),
            ('pca',PCA())
        ])
ct = ColumnTransformer([
    ('numeric',num_pl,num_cols)
],remainder='passthrough')
ct

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""

In [8]:
def objective(trial):
    pl = Pipeline([
        ('preprocess',clone(ct)),
        ('estimator',KMeans(n_clusters=3))
    ])
    params = {
        'preprocess__numeric__aslt__beta': trial.suggest_float('preprocess__numeric__aslt__beta',-50,50),
        'preprocess__numeric__pca__n_components': trial.suggest_int('preprocess__numeric__pca__n_components',2,num_cols.shape[0])
    }
    pl.set_params(**params)

    clusters = pl.fit_predict(df_sex)
    ari_score = adjusted_rand_score(y_sex,clusters)
    return ari_score

study_kmeans = optuna.create_study(direction='maximize')
study_kmeans.optimize(objective,n_trials=50)

In [9]:
def objective(trial):
    pl = Pipeline([
        ('preprocess',clone(ct)),
        ('estimator',SpectralClustering(n_clusters=3))
    ])
    params = {
        'preprocess__numeric__aslt__beta': trial.suggest_float('preprocess__numeric__aslt__beta',-50,50),
        'preprocess__numeric__pca__n_components': trial.suggest_int('preprocess__numeric__pca__n_components',2,num_cols.shape[0]),
        'estimator__n_components': trial.suggest_int('estimator__n_components',df_out.shape[1]-1,20),
        'estimator__affinity': trial.suggest_categorical('estimator__affinity',['rbf','nearest_neighbors']),
        'estimator__assign_labels': trial.suggest_categorical('estimator__assign_labels',['kmeans','discretize','cluster_qr'])
    }
    pl.set_params(**params)

    clusters = pl.fit_predict(df_sex)
    ari_score = adjusted_rand_score(y_sex,clusters)
    return ari_score

study_sc = optuna.create_study(direction='maximize')
study_sc.optimize(objective,n_trials=50)

In [10]:
def objective(trial):
    pl = Pipeline([
        ('preprocess',clone(ct)),
        ('estimator',AgglomerativeClustering(n_clusters=3))
    ])
    params = {
        'preprocess__numeric__aslt__beta': trial.suggest_float('preprocess__numeric__aslt__beta',-50,50),
        'preprocess__numeric__pca__n_components': trial.suggest_int('preprocess__numeric__pca__n_components',2,num_cols.shape[0]),
        'estimator__metric': trial.suggest_categorical('estimator__metric',['euclidean','l1','l2','cosine']),
        'estimator__linkage': trial.suggest_categorical('estimator__linkage',['complete','average','single'])
    }
    pl.set_params(**params)

    clusters = pl.fit_predict(df_sex)
    ari_score = adjusted_rand_score(y_sex,clusters)
    return ari_score

study_hc = optuna.create_study(direction='maximize')
study_hc.optimize(objective,n_trials=50)

In [11]:
def objective(trial):
    pl = Pipeline([
        ('preprocess',clone(ct)),
        ('estimator',AgglomerativeClustering(n_clusters=3))
    ])
    params = {
        'preprocess__numeric__aslt__beta': trial.suggest_float('preprocess__numeric__aslt__beta',-50,50),
        'preprocess__numeric__pca__n_components': trial.suggest_int('preprocess__numeric__pca__n_components',2,num_cols.shape[0]),
        'estimator__metric': trial.suggest_categorical('estimator__metric',['euclidean']),
        'estimator__linkage': trial.suggest_categorical('estimator__linkage',['ward'])
    }
    pl.set_params(**params)

    clusters = pl.fit_predict(df_sex)
    ari_score = adjusted_rand_score(y_sex,clusters)
    return ari_score

study_hcw = optuna.create_study(direction='maximize')
study_hcw.optimize(objective,n_trials=50)

In [14]:
results_sex = {
    'kmeans':study_kmeans.best_params,
    'sc':study_sc.best_params,
    'hc':study_hc.best_params if study_hc.best_value > study_hcw.best_value else study_hcw.best_params   
}

results_sex

{'kmeans': {'preprocess__numeric__aslt__beta': 5.273496665560491,
  'preprocess__numeric__pca__n_components': 5},
 'sc': {'preprocess__numeric__aslt__beta': -9.970777238943953,
  'preprocess__numeric__pca__n_components': 2,
  'estimator__n_components': 17,
  'estimator__affinity': 'rbf',
  'estimator__assign_labels': 'discretize'},
 'hc': {'preprocess__numeric__aslt__beta': 5.526874989120124,
  'preprocess__numeric__pca__n_components': 8,
  'estimator__metric': 'cosine',
  'estimator__linkage': 'average'}}

In [13]:
asffsd

NameError: name 'asffsd' is not defined

### **Check for Age/Rings**

In [ ]:
df_ring = df_out.drop('rings',axis=1)
y_ring = df_out['rings']
y_ring = np.digitize(y_ring,[0,5,10,15,20,25,30])

num_cols = df_ring.select_dtypes(include='number').columns
cat_cols = df_ring.select_dtypes(exclude='number').columns
num_pl = Pipeline([
            ('aslt',ASLT(beta=10)),
            ('scaler',StandardScaler()),
            ('pca',PCA())
        ])
ct = ColumnTransformer([
    ('numeric',num_pl,num_cols),
    ('categorical',OrdinalEncoder(),cat_cols)
],remainder='passthrough')
ct

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``fea

In [ ]:
def objective(trial):
    pl = Pipeline([
        ('preprocess',clone(ct)),
        ('estimator',KMeans(n_clusters=6))
    ])
    params = {
        'preprocess__numeric__aslt__beta': trial.suggest_float('preprocess__numeric__aslt__beta',-50,50),
        'preprocess__numeric__pca__n_components': trial.suggest_int('preprocess__numeric__pca__n_components',2,num_cols.shape[0])
    }
    pl.set_params(**params)

    clusters = pl.fit_predict(df_ring)
    ari_score = adjusted_rand_score(y_ring,clusters)
    return ari_score

study_kmeans = optuna.create_study(direction='maximize')
study_kmeans.optimize(objective,n_trials=50)

[I 2026-11-17 22:31:58,455] A new study created in memory with name: no-name-d9527204-bd13-4079-afb7-d026d1716742
[I 2026-11-17 22:31:58,468] Trial 0 finished with value: 0.10533191536969777 and parameters: {'preprocess__numeric__aslt__beta': -43.542906023557805, 'preprocess__numeric__pca__n_components': 4}. Best is trial 0 with value: 0.10533191536969777.
[I 2026-11-17 22:31:58,480] Trial 1 finished with value: 0.07470060783769893 and parameters: {'preprocess__numeric__aslt__beta': 9.53787193408688, 'preprocess__numeric__pca__n_components': 7}. Best is trial 0 with value: 0.10533191536969777.
[I 2026-11-17 22:31:58,496] Trial 2 finished with value: 0.07942769787588402 and parameters: {'preprocess__numeric__aslt__beta': -3.9074439039775797, 'preprocess__numeric__pca__n_components': 3}. Best is trial 0 with value: 0.10533191536969777.
[I 2026-11-17 22:31:58,507] Trial 3 finished with value: 0.07218330694642117 and parameters: {'preprocess__numeric__aslt__beta': 33.20995399530095, 'prepr

In [ ]:
def objective(trial):
    pl = Pipeline([
        ('preprocess',clone(ct)),
        ('estimator',SpectralClustering(n_clusters=6))
    ])
    params = {
        'preprocess__numeric__aslt__beta': trial.suggest_float('preprocess__numeric__aslt__beta',-50,50),
        'preprocess__numeric__pca__n_components': trial.suggest_int('preprocess__numeric__pca__n_components',2,num_cols.shape[0]),
        'estimator__n_components': trial.suggest_int('estimator__n_components',df_out.shape[1]-1,20),
        'estimator__affinity': trial.suggest_categorical('estimator__affinity',['rbf','nearest_neighbors']),
        'estimator__assign_labels': trial.suggest_categorical('estimator__assign_labels',['kmeans','discretize','cluster_qr'])
    }
    pl.set_params(**params)

    clusters = pl.fit_predict(df_ring)
    ari_score = adjusted_rand_score(y_ring,clusters)
    return ari_score

study_sc = optuna.create_study(direction='maximize')
study_sc.optimize(objective,n_trials=50)

[I 2026-11-17 22:31:59,260] A new study created in memory with name: no-name-e32bbe41-d7dd-417d-948e-d319c05450ef
[I 2026-11-17 22:32:00,555] Trial 0 finished with value: 0.09986588226872399 and parameters: {'preprocess__numeric__aslt__beta': -13.436220125001341, 'preprocess__numeric__pca__n_components': 5, 'estimator__n_components': 13, 'estimator__affinity': 'rbf', 'estimator__assign_labels': 'cluster_qr'}. Best is trial 0 with value: 0.09986588226872399.
[I 2026-11-17 22:32:00,884] Trial 1 finished with value: 0.0743589273176966 and parameters: {'preprocess__numeric__aslt__beta': 5.002197094552798, 'preprocess__numeric__pca__n_components': 2, 'estimator__n_components': 9, 'estimator__affinity': 'nearest_neighbors', 'estimator__assign_labels': 'kmeans'}. Best is trial 0 with value: 0.09986588226872399.
[I 2026-11-17 22:32:02,189] Trial 2 finished with value: 0.05139653652319633 and parameters: {'preprocess__numeric__aslt__beta': 47.57505202640239, 'preprocess__numeric__pca__n_compone

In [ ]:
def objective(trial):
    pl = Pipeline([
        ('preprocess',clone(ct)),
        ('estimator',AgglomerativeClustering(n_clusters=6))
    ])
    params = {
        'preprocess__numeric__aslt__beta': trial.suggest_float('preprocess__numeric__aslt__beta',-50,50),
        'preprocess__numeric__pca__n_components': trial.suggest_int('preprocess__numeric__pca__n_components',2,num_cols.shape[0]),
        'estimator__metric': trial.suggest_categorical('estimator__metric',['euclidean','l1','l2','cosine']),
        'estimator__linkage': trial.suggest_categorical('estimator__linkage',['complete','average','single'])
    }
    pl.set_params(**params)

    clusters = pl.fit_predict(df_ring)
    ari_score = adjusted_rand_score(y_ring,clusters)
    return ari_score

study_hc = optuna.create_study(direction='maximize')
study_hc.optimize(objective,n_trials=50)

[I 2026-11-17 22:32:53,667] A new study created in memory with name: no-name-424a4143-582f-4064-baf6-17ca731a9f82
[I 2026-11-17 22:32:53,732] Trial 0 finished with value: 0.0006393723267762217 and parameters: {'preprocess__numeric__aslt__beta': -9.433915447693607, 'preprocess__numeric__pca__n_components': 4, 'estimator__metric': 'l2', 'estimator__linkage': 'single'}. Best is trial 0 with value: 0.0006393723267762217.
[I 2026-11-17 22:32:53,918] Trial 1 finished with value: 0.13604921405073758 and parameters: {'preprocess__numeric__aslt__beta': -33.317796836738694, 'preprocess__numeric__pca__n_components': 2, 'estimator__metric': 'cosine', 'estimator__linkage': 'complete'}. Best is trial 1 with value: 0.13604921405073758.
[I 2026-11-17 22:32:54,107] Trial 2 finished with value: 0.06215057089094149 and parameters: {'preprocess__numeric__aslt__beta': -39.63440082537467, 'preprocess__numeric__pca__n_components': 2, 'estimator__metric': 'l1', 'estimator__linkage': 'complete'}. Best is trial

In [ ]:
def objective(trial):
    pl = Pipeline([
        ('preprocess',clone(ct)),
        ('estimator',AgglomerativeClustering(n_clusters=6))
    ])
    params = {
        'preprocess__numeric__aslt__beta': trial.suggest_float('preprocess__numeric__aslt__beta',-50,50),
        'preprocess__numeric__pca__n_components': trial.suggest_int('preprocess__numeric__pca__n_components',2,num_cols.shape[0]),
        'estimator__metric': trial.suggest_categorical('estimator__metric',['euclidean']),
        'estimator__linkage': trial.suggest_categorical('estimator__linkage',['ward'])
    }
    pl.set_params(**params)

    clusters = pl.fit_predict(df_ring)
    ari_score = adjusted_rand_score(y_ring,clusters)
    return ari_score

study_hcw = optuna.create_study(direction='maximize')
study_hcw.optimize(objective,n_trials=50)

[I 2026-11-17 22:33:03,329] A new study created in memory with name: no-name-bc3e2f10-8942-49fc-ae79-321980a6bdc6
[I 2026-11-17 22:33:03,538] Trial 0 finished with value: 0.0893608467077816 and parameters: {'preprocess__numeric__aslt__beta': 21.237632748071007, 'preprocess__numeric__pca__n_components': 2, 'estimator__metric': 'euclidean', 'estimator__linkage': 'ward'}. Best is trial 0 with value: 0.0893608467077816.
[I 2026-11-17 22:33:03,787] Trial 1 finished with value: 0.07954068102956018 and parameters: {'preprocess__numeric__aslt__beta': -17.562566413841786, 'preprocess__numeric__pca__n_components': 6, 'estimator__metric': 'euclidean', 'estimator__linkage': 'ward'}. Best is trial 0 with value: 0.0893608467077816.
[I 2026-11-17 22:33:03,996] Trial 2 finished with value: 0.07405602601064742 and parameters: {'preprocess__numeric__aslt__beta': 49.1333085435208, 'preprocess__numeric__pca__n_components': 2, 'estimator__metric': 'euclidean', 'estimator__linkage': 'ward'}. Best is trial 0

In [ ]:
results_ring = {
    'kmeans':study_kmeans.best_params,
    'sc':study_sc.best_params,
    'hc':study_hc.best_params if study_hc.best_value > study_hcw.best_value else study_hcw.best_params   
}

results_ring

{'kmeans': [0.10961904494611685,
  {'preprocess__numeric__aslt__beta': -35.05735706559091,
   'preprocess__numeric__pca__n_components': 5}],
 'sc': [0.16178765249514582,
  {'preprocess__numeric__aslt__beta': -34.29252870853722,
   'preprocess__numeric__pca__n_components': 3,
   'estimator__n_components': 8,
   'estimator__affinity': 'rbf',
   'estimator__assign_labels': 'cluster_qr'}],
 'hc': [0.1769480221974349,
  {'preprocess__numeric__aslt__beta': -16.92313624162334,
   'preprocess__numeric__pca__n_components': 2,
   'estimator__metric': 'cosine',
   'estimator__linkage': 'average'}]}

# **Visualization**